In [1]:
import sys

import numpy as np
import torch
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, WeightedRandomSampler

import internal.utils.telegram_utils as tg
from internal.data_types import PainDataset
from internal.nn.models.pain_tcn_bilstm_attn import PainTCNBiLSTMAttn
from internal.nn.training.pain_tcn_bilstm_attn_trainer import PainTCNBiLSTMAttnTrainer
from internal.persistence_manager import PersistenceManager

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
@torch.no_grad()
def infer_logits(_model, _x_num, _x_surv, _x_sta, _x_summ, batch_size=64):
    _model.eval()
    ds = PainDataset(_x_num, _x_surv, _x_sta, _x_summ)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    predictions = []
    for b in dl:
        x_num  = b['x_num'].to(device)
        x_sta  = b['x_sta'].to(device)
        x_surv = [s.to(device) for s in b['x_surv']]
        x_summ = b['x_summ'].to(device)
        logits = _model(x_num, x_surv, x_sta, x_summ)  # (B,3)
        predictions.append(logits.detach().cpu().numpy())
    return np.concatenate(predictions, axis=0)  # (N,3)

@torch.no_grad()
def infer_logits_tta(_model, _x_num, _x_surv, _x_sta, _x_summ, _n_aug=5, _noise_std=0.01, _roll=2) -> np.ndarray:
    _model.eval()
    outs = []
    for _k in range(_n_aug):
        xn = _x_num.copy()
        if _k > 0:
            # small Gaussian jitter on numeric channels
            xn += np.random.normal(0, _noise_std, size=xn.shape).astype(xn.dtype)
            # tiny circular time shift
            shift = np.random.randint(-_roll, _roll + 1)
            if shift != 0:
                xn = np.roll(xn, shift=shift, axis=1)  # roll along the time
        outs.append(infer_logits(_model, xn, _x_surv, _x_sta, _x_summ, batch_size=64))
    return np.mean(outs, axis=0)  # (N,3)

@torch.no_grad()
def collect_val_predictions(_model, _val_loader):
    _model.eval()
    y_true_list = []
    y_pred_list = []

    for batch in _val_loader:
        x_num  = batch['x_num'].to(device)
        x_sta  = batch['x_sta'].to(device)
        x_surv = [s.to(device) for s in batch['x_surv']]
        x_summ = batch['x_summ'].to(device)
        y      = batch['y'].to(device)

        logits = _model(x_num, x_surv, x_sta, x_summ)
        y_hat  = logits.argmax(dim=1)

        y_true_list.append(y.cpu().numpy())
        y_pred_list.append(y_hat.cpu().numpy())

    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)
    return y_true, y_pred

In [3]:
def smooth_sequence_majority(_seq: np.ndarray, _k: int = 3) -> np.ndarray:
    """
    seq: 1D array of length T with integer levels {0,1,2}
    k: window size (odd), e.g. 3 or 5
    """
    assert _k % 2 == 1, "Window size k must be odd"
    T = len(_seq)
    out = _seq.copy()
    half = _k // 2

    for t in range(T):
        left = max(0, t - half)
        right = min(T, t + half + 1)
        window = _seq[left:right]
        vals, counts = np.unique(window, return_counts=True)
        out[t] = vals[np.argmax(counts)]  # most frequent value in the window

    return out


def smooth_surveys_majority(_x_surv: np.ndarray, _k: int = 3) -> np.ndarray:
    """
    X_surv: array of shape (N, T) with integer levels {0,1,2}
    Apply majority smoothing along time for each patient.
    """
    N, T = _x_surv.shape
    X_smooth = np.empty_like(_x_surv)
    for i in range(N):
        X_smooth[i] = smooth_sequence_majority(_x_surv[i], _k=_k)
    return X_smooth

def smooth_surveys_numeric(_s: np.ndarray, _k: int = 3) -> np.ndarray:
    """Smooth survey curve with a centered moving average (float output)."""
    S = _s.astype(np.float32)
    N, T = S.shape
    pad = _k // 2
    kernel = np.ones(_k, dtype=np.float32) / _k

    # pad along time axis only
    padded = np.pad(S, ((0, 0), (pad, pad)), mode='edge')  # (N, T + 2*pad)
    out = np.empty_like(S, dtype=np.float32)

    for i in range(N):
        out[i] = np.convolve(padded[i], kernel, mode='valid')

    return out


In [4]:
# TODO: remove this debug code later
# --- Debug ---
import matplotlib.pyplot as plt
import pandas as pd

def extract_patient(_pid, _pid_all, _x_num_all, _x_surv_all, _x_sta_all, _x_summ_all, _y_all):
    idxs = np.where(_pid_all == _pid)[0]
    assert len(idxs) == 1, "Each patient must appear exactly once!"
    i = idxs[0]
    return {
        "pid": _pid,
        "y_true": int(_y_all[i]),
        "X_num": _x_num_all[i],     # (160, 90)
        "X_surv": [x[i] for x in _x_surv_all],  # 4 arrays (160,)
        "X_sta": _x_sta_all[i],     # (7,)
        "X_summ": _x_summ_all[i],   # (F_summ,)
    }

def plot_surveys(patient):
    surveys = patient["X_surv"]
    _pid = patient["pid"]

    plt.figure(figsize=(12,6))
    for i, s in enumerate(surveys, 1):
        plt.plot(s, label=f"survey_{i}")
    plt.title(f"Survey curves for patient {_pid}")
    plt.legend()
    plt.grid(True)
    plt.savefig(
        'figures/patient_{}_surveys.pdf'.format(_pid)
    )
    # plt.show()

def plot_static_features(_patient, _static_names=None):
    sta = _patient["X_sta"]
    _pid = _patient["pid"]

    if _static_names is None:
        _static_names = [f"sta_{i}" for i in range(len(sta))]

    plt.figure(figsize=(10,4))
    plt.bar(_static_names, sta)
    plt.title(f"Static features for patient {_pid}")
    plt.xticks(rotation=45)
    plt.grid(True, axis="y")
    plt.savefig(
        'figures/patient_{}_static_features.pdf'.format(_pid)
    )
    # plt.show()

def plot_umap(_patient):
    try:
        import umap
    except ImportError:
        print("UMAP not installed, skipping UMAP plot.")
        return
    _x = _patient["X_num"]  # shape (160, 90)
    _pid = _patient["pid"]

    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="euclidean")
    emb = reducer.fit_transform(_x)

    plt.figure(figsize=(8,6))
    plt.scatter(emb[:,0], emb[:,1], c=np.arange(len(emb)), cmap="viridis")
    plt.colorbar(label="timestep")
    plt.title(f"UMAP embeddings over time for patient {_pid}")
    plt.savefig(
        'figures/patient_{}_umap.pdf'.format(_pid)
    )
    # plt.show()

def plot_summary(_patient):
    summ = _patient["X_summ"]
    _pid = _patient["pid"]

    names = [f"summ_{i}" for i in range(len(summ))]

    plt.figure(figsize=(12,4))
    plt.bar(names, summ)
    plt.title(f"Summary features for patient {_pid}")
    plt.xticks(rotation=90)
    plt.grid(True, axis="y")
    plt.savefig(
        'figures/patient_{}_summary_features.pdf'.format(_pid)
    )
    # plt.show()


In [5]:
# load data
data = PersistenceManager.load_arrays_v2()

# set the seed for reproducibility
seed = 2021
np.random.seed(seed)
torch.manual_seed(seed)

# prepare stratified group K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
pid_all     = np.concatenate([data.pid_train,        data.pid_val], axis=0)
y_all       = np.concatenate([data.y_train,          data.y_val],   axis=0)
X_num_all   = np.concatenate([data.X_dyn_num_train,  data.X_dyn_num_val],  axis=0)
X_sta_all   = np.concatenate([data.X_sta_train,      data.X_sta_val],      axis=0)
X_surv_all  = [np.concatenate([data.X_surv_train[i], data.X_surv_val[i]], axis=0) for i in range(4)]
X_surv_all_smooth = X_surv_all # [smooth_surveys_numeric(s, _k=5) for s in X_surv_all]
X_summ_all  = np.concatenate([data.X_dyn_summ_train, data.X_dyn_summ_val], axis=0)

# range of class-2 multipliers to try
class2_multipliers = [0.6, 0.7, 0.8, 0.9, 1]  # [1, 1.05, 1.1, 1.15, 1.2]
# to store per-multiplier CV stats
mult_stats: dict[float, dict[str, float]] = {}

# trainer instance
trainer = PainTCNBiLSTMAttnTrainer(
    classes=[0, 1, 2],
    epochs=1000,
    patience=28,
    warmup_epochs=20,
    p_drop_summ=0.3,
    summary_window=data.feat_eng.window,
    device=device
)

# notify me about phase A start
tg.send_message("🚀 Phase A: Searching best class-2 multiplier via CV...", raise_on_failure=False)

# for each class-2 multiplier, do a full CV and record mean/std
for class2_multiplier in class2_multipliers:
    # to store fold scores for this multiplier
    fold_scores_m = []

    # K-Fold CV loop - no test logits here
    for k, (tr_idx, va_idx) in enumerate(skf.split(X=np.zeros(len(y_all)),
                                                   y=y_all,
                                                   groups=pid_all)):
        print(f"=== [Phase A] Fold {k} (grouped by patient) with class-2 multiplier {class2_multiplier} ===")

        # --- prepare datasets ---
        Xtr_num, Xva_num    = X_num_all[tr_idx], X_num_all[va_idx]
        Xtr_sta, Xva_sta    = X_sta_all[tr_idx], X_sta_all[va_idx]
        Xtr_surv = [s[tr_idx] for s in X_surv_all_smooth]
        Xva_surv = [s[va_idx] for s in X_surv_all_smooth]
        Xtr_summ, Xva_summ  = X_summ_all[tr_idx], X_summ_all[va_idx]
        ytr, yva            = y_all[tr_idx],      y_all[va_idx]
        pid_tr, pid_va      = pid_all[tr_idx],    pid_all[va_idx]

        # --- sanity check: no leakage ---
        assert set(pid_tr).isdisjoint(set(pid_va)), "Patient leaked between train and val!"

        # --- create datasets ---
        train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
        val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

        # --- create weighted sampler for class balancing ---
        class_counts = np.bincount(ytr, minlength=3).astype(np.float32)
        inv = 1.0 / np.clip(class_counts, 1, None)
        inv = inv / inv.mean()
        alpha = 0.5
        w_class = np.power(inv, alpha)
        cap = 5.0
        w_class = np.minimum(w_class, cap * w_class.mean())
        sample_w = w_class[ytr]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_w, dtype=torch.double),
            num_samples=len(ytr),
            replacement=True
        )

        # --- create data loaders that use the sampler ---
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, sampler=sampler, num_workers=4, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

        # -- calculate priors from training fold ---
        counts_tr = np.bincount(ytr, minlength=3).astype(np.float32)
        priors_tr = counts_tr / counts_tr.sum()

        # --- Train on this fold ---
        model, f1, best_b, t_fold, best_tau = trainer.train_one_fold(
            model=PainTCNBiLSTMAttn(
                d_num=90,
                d_emb=4,
                d_sta=7,
                d_summ=data.X_dyn_summ_train.shape[1],
                tcn_channels=128,
                lstm_hidden=128,
                num_classes=3,
                dropout=0.3
            ).to(device),
            k=k,
            train_loader=train_loader,
            val_loader=val_loader,
            y_train=ytr,
            y_val=yva,
            priors=priors_tr,
            class_multipliers={2: class2_multiplier}
        )

        # record fold score
        fold_scores_m.append(f1)

    # after all folds, compute mean/std for this multiplier
    mean_m = float(np.mean(fold_scores_m))
    std_m  = float(np.std(fold_scores_m))
    mult_stats[class2_multiplier] = {'mean': mean_m, 'std': std_m}

    # notify me about this multiplier's CV result
    print(f"[Phase A] mul={class2_multiplier}: mean={mean_m:.4f} std={std_m:.4f}")
    tg.send_message(
        f"📊 [Phase A] class2_multiplier={class2_multiplier} → mean F1={mean_m:.4f}, std={std_m:.4f}",
        raise_on_failure=False
    )

# after all multipliers, print summary
print("=== [Phase A] Class-2 Multiplier Search Complete ===")
print("Per-multiplier CV:")
for m in class2_multipliers:
    print(f"  mul={m}: mean={mult_stats[m]['mean']:.4f} std={mult_stats[m]['std']:.4f}")

best_mult = max(mult_stats.keys(), key=lambda m: mult_stats[m]['mean'])
print("Selected best multiplier:", best_mult, "with mean F1:", mult_stats[best_mult]['mean'])
tg.send_message(
    f"✅ [Phase A] Best class2_multiplier={best_mult} with mean F1={mult_stats[best_mult]['mean']:.4f}",
    raise_on_failure=False
)

Arrays v2 loaded successfully from: /home/andre/university/AN2DL-Challenge-1/notebooks/processed/arrays_v2.joblib
=== [Phase A] Fold 0 (grouped by patient) with class-2 multiplier 0.6 ===
[F0 001] t_loss=0.3983 | F1(macro)=0.4277 | rec=[0.804 0.05  0.8  ] | lr=1.00e-03 | patience=1/28
[F0 002] t_loss=0.2882 | F1(macro)=0.4815 | rec=[0.873 0.1   0.8  ] | lr=1.00e-03 | patience=1/28
[F0 003] t_loss=0.2586 | F1(macro)=0.7708 | rec=[0.882 0.8   0.8  ] | lr=1.00e-03 | patience=1/28
[F0 004] t_loss=0.1790 | F1(macro)=0.8419 | rec=[0.98 0.7  0.9 ] | lr=1.00e-03 | patience=1/28
[F0 005] t_loss=0.1396 | F1(macro)=0.7276 | rec=[0.902 0.6   0.9  ] | lr=1.00e-03 | patience=1/28
[F0 006] t_loss=0.1056 | F1(macro)=0.7999 | rec=[0.931 0.65  1.   ] | lr=1.00e-03 | patience=2/28
[F0 007] t_loss=0.0709 | F1(macro)=0.8558 | rec=[0.971 0.75  0.8  ] | lr=1.00e-03 | patience=3/28
[F0 008] t_loss=0.0952 | F1(macro)=0.8901 | rec=[0.951 0.85  1.   ] | lr=1.00e-03 | patience=1/28
[F0 009] t_loss=0.0682 | F1(mac

False

In [6]:
# --- Phase B: Final training with the best multiplier and test ensembling ---

# to store test logits from each fold
test_fold_logits = []
# to store final fold scores
fold_scores_final = []
# to store temperatures within final folds
temperatures = []
# to store taus within final folds
taus = []

# notify me about phase B start
tg.send_message(
    f"🚀 Phase B: Final training with best class2_multiplier={best_mult} and test ensembling...",
    raise_on_failure=False
)

# redirect print into file log/log_phase_b.log
class Logger(object):
    def __init__(self, filename="log/log_phase_b.log"):
        self.terminal = sys.stdout
        self.log = open(filename, "a")
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        pass
sys.stdout = Logger("log/log_phase_b.log")

# K-Fold CV loop with best multiplier and test logits
for k, (tr_idx, va_idx) in enumerate(skf.split(X=np.zeros(len(y_all)),
                                               y=y_all,
                                               groups=pid_all)):
    print(f"=== [Phase B] Fold {k} (grouped by patient) with class-2 multiplier {best_mult} ===")

    # --- prepare datasets ---
    Xtr_num, Xva_num    = X_num_all[tr_idx], X_num_all[va_idx]
    Xtr_sta, Xva_sta    = X_sta_all[tr_idx], X_sta_all[va_idx]
    Xtr_surv            = [s[tr_idx] for s in X_surv_all]
    Xva_surv            = [s[va_idx] for s in X_surv_all]
    Xtr_summ, Xva_summ  = X_summ_all[tr_idx], X_summ_all[va_idx]
    ytr, yva            = y_all[tr_idx],      y_all[va_idx]
    pid_tr, pid_va      = pid_all[tr_idx],    pid_all[va_idx]

    # --- sanity check: no leakage ---
    assert set(pid_tr).isdisjoint(set(pid_va)), "Patient leaked between train and val!"

    # --- create datasets ---
    train_ds = PainDataset(Xtr_num, Xtr_surv, Xtr_sta, Xtr_summ, ytr)
    val_ds   = PainDataset(Xva_num, Xva_surv, Xva_sta, Xva_summ, yva)

    # --- create weighted sampler for class balancing ---
    class_counts = np.bincount(ytr, minlength=3).astype(np.float32)
    inv = 1.0 / np.clip(class_counts, 1, None)
    inv = inv / inv.mean()
    alpha = 0.5
    w_class = np.power(inv, alpha)
    cap = 5.0
    w_class = np.minimum(w_class, cap * w_class.mean())
    sample_w = w_class[ytr]
    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_w, dtype=torch.double),
        num_samples=len(ytr),
        replacement=True
    )

    # --- create data loaders that use the sampler ---
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    # -- calculate priors from training fold ---
    counts_tr = np.bincount(ytr, minlength=3).astype(np.float32)
    priors_tr = counts_tr / counts_tr.sum()

    # --- Train on this fold ---
    model, f1, best_b, t_fold, best_tau = trainer.train_one_fold(
        model=PainTCNBiLSTMAttn(
            d_num=90,
            d_emb=4,
            d_sta=7,
            d_summ=data.X_dyn_summ_train.shape[1],
            tcn_channels=128,
            lstm_hidden=128,
            num_classes=3,
            dropout=0.3
        ).to(device),
        k=k,
        train_loader=train_loader,
        val_loader=val_loader,
        y_train=ytr,
        y_val=yva,
        priors=priors_tr,
        class_multipliers={2: best_mult}
    )

    # --- record fold score ---
    fold_scores_final.append(f1)
    temperatures.append(t_fold)
    taus.append(best_tau)

    # ~~~ Debug per-pirate val predictions ~~~
    #    y_true_val, y_pred_val = collect_val_predictions(model, val_loader)
    #    assert len(y_true_val) == len(pid_va)
    #    df_fold = pd.DataFrame({
    #        "pid": pid_va,
    #        "y_true": y_true_val,
    #        "y_pred": y_pred_val,
    #    })
    #    per_pid_f1 = (
    #        df_fold.groupby("pid")
    #               .apply(lambda _g: f1_score(_g.y_true, _g.y_pred,
    #                                          average="macro", labels=[0, 1, 2]))
    #               .sort_values()
    #    )
    #    print(f"\n[DEBUG] Worst patients in Phase B – fold {k}, mul={best_mult}:")
    #    print(per_pid_f1.head(10))
    #    worst_pids = per_pid_f1.head(5).index.tolist()
    #    rows = []
    #    for pid in worst_pids:
    #        g = df_fold[df_fold.pid == pid]
    #        rows.append({
    #            "pid": pid,
    #            "n_samples": len(g),
    #            "f1_macro": f1_score(g.y_true, g.y_pred, average="macro", labels=[0, 1, 2]),
    #            "true_counts": np.bincount(g.y_true, minlength=3).tolist(),
    #            "pred_counts": np.bincount(g.y_pred, minlength=3).tolist(),
    #        })
    #    df_bad = pd.DataFrame(rows)
    #    print(f"\n[DEBUG] Details for worst patients in fold {k}, mul={best_mult}:")
    #    print(df_bad.to_json())
    #    tg.send_message(
    #        "🩻 Worst pids in fold "
    #        f"{k}, mul={best_mult}:\n" +
    #        "\n".join([f"pid={int(pid)}, F1={per_pid_f1[pid]:.3f}" for pid in per_pid_f1.head(5).index]),
    #        raise_on_failure=False
    #    )
    # ~~~ End debug ~~~

    # --- infer test logits with TTA ---
    logits_te = infer_logits_tta(
        model,
        data.X_dyn_num_test,
        data.X_surv_test, # [smooth_surveys_numeric(s, _k=5) for s in data.X_surv_test],
        data.X_sta_test,
        data.X_dyn_summ_test,
        _n_aug=5, _noise_std=0.01, _roll=2
    ).astype(np.float32)

    # --- temperature scaling (per-fold scalar) ---
    logits_te = logits_te / float(t_fold)

    # -- prior correction with the best tau and bias ---
    log_p = np.log(np.clip(priors_tr, 1e-8, 1.0)).astype(np.float32)
    logits_te = logits_te + best_tau * log_p[None, :]
    logits_te = logits_te + best_b[None, :]

    # --- store for ensembling ---
    test_fold_logits.append(logits_te)

# --- ensemble and submission ---
logits_stack = np.stack(test_fold_logits, axis=0)   # (5, N_test, 3)
logits_avg   = logits_stack.mean(axis=0)            # (N_test, 3)
pred_idx     = logits_avg.argmax(axis=1)

idx_to_label = {0:'no_pain', 1:'low_pain', 2:'high_pain'}
pred_labels  = [idx_to_label[int(i)] for i in pred_idx]

ids_test = [f'{i:03}' for i in range(len(pred_labels))]
sub = pd.DataFrame({'sample_index': ids_test, 'label': pred_labels})

_filename = f'PainTCNBiLSTMAttn_bestmul_{best_mult}_seed{seed}'
sub.to_csv(f'submissions/{_filename}.csv', index=False)
print("Saved:", f'submissions/{_filename}.csv')
tg.send_file(
    f'submissions/{_filename}.csv',
    caption=f"✅ Submission from best_mult={best_mult}",
    raise_on_failure=False
)

print(f"[Phase B] Final CV with best_mult={best_mult}: mean F1={np.mean(fold_scores_final):.4f} std={np.std(fold_scores_final):.4f}")
tg.send_message(
    f"✅ [Phase B] Done. CV mean={np.mean(fold_scores_final):.4f} std={np.std(fold_scores_final):.4f}",
    False
)


=== [Phase B] Fold 0 (grouped by patient) with class-2 multiplier 0.7 ===
[F0 001] t_loss=0.4119 | F1(macro)=0.4146 | rec=[0.971 0.    0.4  ] | lr=1.00e-03 | patience=1/28
[F0 002] t_loss=0.3368 | F1(macro)=0.5825 | rec=[0.941 0.2   0.8  ] | lr=1.00e-03 | patience=1/28
[F0 003] t_loss=0.3117 | F1(macro)=0.6802 | rec=[0.961 0.75  0.3  ] | lr=1.00e-03 | patience=1/28
[F0 004] t_loss=0.2226 | F1(macro)=0.6764 | rec=[0.814 0.85  0.6  ] | lr=1.00e-03 | patience=1/28
[F0 005] t_loss=0.1832 | F1(macro)=0.8188 | rec=[0.961 0.7   0.8  ] | lr=1.00e-03 | patience=2/28
[F0 006] t_loss=0.1469 | F1(macro)=0.7949 | rec=[0.941 0.75  0.8  ] | lr=1.00e-03 | patience=1/28
[F0 007] t_loss=0.0950 | F1(macro)=0.7735 | rec=[0.912 0.8   0.8  ] | lr=1.00e-03 | patience=2/28
[F0 008] t_loss=0.0782 | F1(macro)=0.8048 | rec=[0.902 0.8   0.9  ] | lr=1.00e-03 | patience=3/28
[F0 009] t_loss=0.0810 | F1(macro)=0.7701 | rec=[0.863 0.75  1.   ] | lr=1.00e-03 | patience=4/28
[F0 010] t_loss=0.0720 | F1(macro)=0.8634 | 

True

<xml><var name="_dummy_ipython_val"  />
<var name="_dummy_special_var"  />
<var name="_dummy_special_var"  />
<var name="StratifiedGroupKFold" type="ABCMeta" qualifier="abc" value="%3Cclass %27sklearn.model_selection._split.StratifiedGroupKFold%27&gt;" isContainer="True" />
<var name="X_num_all" type="ndarray" qualifier="numpy" value="%5B%5B%5B 1.43780410e%2B00  5.71503818e-01  1.26021230e%2B00 ...  9.99999975e-05%2C    9.99999975e-05  9.99999975e-05%5D%2C  %5B 1.68115854e%2B00  8.12237322e-01  8.28995168e-01 ...  1.34554937e-01%2C    1.20349252e%2B00  2.60257453e-01%5D%2C  %5B 1.03399146e%2B00  5.36631882e-01  1.16868770e%2B00 ...  2.20326990e-01%2C    1.08475697e%2B00  2.98208266e-01%5D%2C  ...%2C  %5B 1.28968894e%2B00  2.30944619e-01  1.03747642e%2B00 ...  4.43848103e-01%2C    9.51917231e-01  5.85379720e-01%5D%2C  %5B 1.26782513e%2B00  8.84079933e-01  9.43937898e-01 ...  4.49626327e-01%2C    9.39136565e-01  5.88176668e-01%5D%2C  %5B 1.28507721e%2B00  4.38795596e-01  8.73604774e-01 .